## This file loads, computes, and presents the results of all of our AU intensity estimation and AU detection methods from Table 2 to Table 6.

In [1]:
import numpy as np
import pickle as pkl
import os
from collections import defaultdict
from IPython.display import display
from config.datasets import *
from config.models import *
from lib.stats import *

In [2]:
# Infomation of all tasks/methods to load
infos = [{"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "IR50", "sub_idx": -2},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-stage4", "sub_idx": -2},
         {"task": "DISFAPlus_AU_intensity_estimation_ninefold", "dataset": "DISFA+", "n_folds": 9, "AUs": DISFAPLUS_AUS, "label_type": "intensity", "model": "IR50", "sub_idx": -3},
         {"task": "DISFAPlus_AU_intensity_estimation_ninefold", "dataset": "DISFA+", "n_folds": 9, "AUs": DISFAPLUS_AUS, "label_type": "intensity", "model": "CSN-IR50-stage4", "sub_idx": -3},
         {"task": "UNBCMcMaster_AU_intensity_estimation_fivefold", "dataset": "UNBCMcMaster", "n_folds": 5, "AUs": UNBCMCMASTER_AUS_IE, "label_type": "intensity", "model": "IR50", "sub_idx": -3},
         {"task": "UNBCMcMaster_AU_intensity_estimation_fivefold", "dataset": "UNBCMcMaster", "n_folds": 5, "AUs": UNBCMCMASTER_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-stage4", "sub_idx": -3},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "IR50", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-stage4", "sub_idx": -2},
         {"task": "DISFAPlus_AU_detection_ninefold", "dataset": "DISFA+", "n_folds": 9, "AUs": DISFAPLUS_AUS, "label_type": "occurrence", "model": "IR50", "sub_idx": -3},
         {"task": "DISFAPlus_AU_detection_ninefold", "dataset": "DISFA+", "n_folds": 9, "AUs": DISFAPLUS_AUS, "label_type": "occurrence", "model": "CSN-IR50-stage4", "sub_idx": -3},
         {"task": "UNBCMcMaster_AU_detection_fivefold", "dataset": "UNBCMcMaster", "n_folds": 5, "AUs": UNBCMCMASTER_AUS_D, "label_type": "occurrence", "model": "IR50", "sub_idx": -3},
         {"task": "UNBCMcMaster_AU_detection_fivefold", "dataset": "UNBCMcMaster", "n_folds": 5, "AUs": UNBCMCMASTER_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-stage4", "sub_idx": -3},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-stage1", "sub_idx": -2},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-stage2", "sub_idx": -2},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-stage3", "sub_idx": -2},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-fc", "sub_idx": -2},
         {"task": "DISFA_AU_intensity_estimation_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_IE, "label_type": "intensity", "model": "CSN-IR50-output", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-stage1", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-stage2", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-stage3", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-fc", "sub_idx": -2},
         {"task": "DISFA_AU_detection_threefold", "dataset": "DISFA", "n_folds": 3, "AUs": DISFA_AUS_D, "label_type": "occurrence", "model": "CSN-IR50-output", "sub_idx": -2}]

In [3]:
for info in infos:
    # Load the data
    all_image_paths = []
    all_pred_labels = np.empty([0, len(info["AUs"])])
    all_true_labels = np.empty([0, len(info["AUs"])])

    for fold in range(1, info["n_folds"]+1):
        path = os.path.join("results", info["task"], info["model"], "results", f"fold-{fold}", f"val_results_epoch{N_EPOCHS}.pkl")
        with open(path, "rb") as f:
            D = pkl.load(f)
            image_paths = [path_pair.split(' ')[-1] for path_pair in D['image_paths']]
            if info["label_type"] == "intensity":
                pred_labels = D['pred_labels_regression']
            elif info["label_type"] == "occurrence":
                pred_labels = D['pred_labels']
            true_labels = D['true_labels']
            all_image_paths.extend(image_paths)
            all_pred_labels = np.concatenate([all_pred_labels, pred_labels])
            all_true_labels = np.concatenate([all_true_labels, true_labels])
    
    # Find out the indices for each participant in the results
    subject_indices = defaultdict(list)
    for index, path in enumerate(all_image_paths):
        subject_indices[path.split('/')[info["sub_idx"]]].append(index)
    sub_IDs = list(subject_indices.keys())

    # Compute the evaluation metrics from the results
    if info["label_type"] == "intensity":
        df = compute_results_for_intensity_estimation(info, all_pred_labels, all_true_labels, subject_indices, sub_IDs)
    elif info["label_type"] == "occurrence":
        df = compute_results_for_occurrence_detection(info, all_pred_labels, all_true_labels, subject_indices)

    # Display the results
    print(f"{info["task"]}    {info["model"]}")
    display(df)

    # For IR50, also compute and present the results for one-frame calibration with baseline subtraction
    if info["model"] == "IR50":
        if info["dataset"] == "DISFA":
            cam_angles = ['Left' if 'Left' in p else 'Right' if 'Right' in p else None for p in all_image_paths]
            subjects = [p.split('/')[info["sub_idx"]] for p in all_image_paths]
            combs = list(zip(cam_angles, subjects))
            combs_unique = list(set(combs))
            baseline_idx_dict = {comb: [(comb[0] in path) and (comb[1] in path) and (f"{DISFA_BASELINE_FRAMES[comb[1]]}.jpg" in path.split('/')) for path in all_image_paths].index(True) for comb in combs_unique}
            baseline_idx = [baseline_idx_dict[comb] for comb in combs]
            baselined_all_pred_labels = all_pred_labels - all_pred_labels[baseline_idx,:]
        elif info["dataset"] == "DISFA+":
            subjects = [p.split('/')[info["sub_idx"]] for p in all_image_paths]
            baseline_idx_dict = {sub: [(sub in path) and (DISFAPLUS_BASELINE_FRAMES[sub] in path) for path in all_image_paths].index(True) for sub in list(subject_indices.keys())}
            baseline_idx = [baseline_idx_dict[sub] for sub in subjects]
            baselined_all_pred_labels = all_pred_labels - all_pred_labels[baseline_idx,:]
        elif info["dataset"] == "UNBCMcMaster":
            subjects = [p.split('/')[info["sub_idx"]] for p in all_image_paths]
            baseline_idx_dict = {sub: [(sub in path) and (UNBCMCMASTER_BASELINE_FRAMES[sub] in path) for path in all_image_paths].index(True) for sub in list(subject_indices.keys())}
            baseline_idx = [baseline_idx_dict[sub] for sub in subjects]
            baselined_all_pred_labels = all_pred_labels - all_pred_labels[baseline_idx,:]

        if info["label_type"] == "intensity":
            df = compute_results_for_intensity_estimation(info, baselined_all_pred_labels, all_true_labels, subject_indices, sub_IDs)
        else:
            df = compute_results_for_occurrence_detection(info, baselined_all_pred_labels, all_true_labels, subject_indices)

        print(f"{info["task"]}    {info["model"]}    One-Frame Calibration with Baseline Subtraction")
        display(df)

PCCs: .53 & .45 & .75 & .62 & .58 & .60 & .84 & .46 & .47 & .25 & .94 & .66 & .60
acrossICCs: .53 & .45 & .75 & .62 & .55 & .57 & .84 & .42 & .47 & .24 & .93 & .65 & .59
withinICCs: .40 & .35 & .66 & .38 & .54 & .46 & .83 & .27 & .45 & .26 & .93 & .57 & .51
MAEs: .37 & .39 & .44 & .11 & .35 & .21 & .34 & .20 & .39 & .21 & .32 & .42 & .31
MSEs: .47 & .57 & .58 & .05 & .37 & .21 & .28 & .13 & .41 & .18 & .21 & .38 & .32
DISFA_AU_intensity_estimation_threefold    IR50


,PCC,"across-participant ICC(3,1)","within-participant ICC(3,1)",MAE,MSE
AU,,,,,
1,0.53,0.53,0.40,0.37,0.47
2,0.45,0.45,0.35,0.39,0.57
4,0.75,0.75,0.66,0.44,0.58
5,0.62,0.62,0.38,0.11,0.05
6,0.58,0.55,0.54,0.35,0.37
9,0.60,0.57,0.46,0.21,0.21
12,0.84,0.84,0.83,0.34,0.28
15,0.46,0.42,0.27,0.20,0.13
17,0.47,0.47,0.45,0.39,0.41


PCCs: .63 & .51 & .76 & .56 & .64 & .63 & .83 & .45 & .45 & .20 & .94 & .69 & .61
acrossICCs: .62 & .51 & .75 & .56 & .60 & .59 & .82 & .39 & .45 & .20 & .93 & .68 & .59
withinICCs: .40 & .35 & .66 & .38 & .54 & .46 & .83 & .27 & .44 & .26 & .93 & .57 & .51
MAEs: .31 & .36 & .41 & .14 & .33 & .20 & .39 & .18 & .37 & .29 & .33 & .38 & .31
MSEs: .37 & .49 & .53 & .07 & .32 & .20 & .33 & .13 & .37 & .31 & .25 & .33 & .31
DISFA_AU_intensity_estimation_threefold    IR50    One-Frame Calibration with Baseline Subtraction


,PCC,"across-participant ICC(3,1)","within-participant ICC(3,1)",MAE,MSE
AU,,,,,
1,0.63,0.62,0.40,0.31,0.37
2,0.51,0.51,0.35,0.36,0.49
4,0.76,0.75,0.66,0.41,0.53
5,0.56,0.56,0.38,0.14,0.07
6,0.64,0.60,0.54,0.33,0.32
9,0.63,0.59,0.46,0.20,0.20
12,0.83,0.82,0.83,0.39,0.33
15,0.45,0.39,0.27,0.18,0.13
17,0.45,0.45,0.44,0.37,0.37


PCCs: .79 & .75 & .81 & .72 & .70 & .63 & .86 & .36 & .57 & .43 & .94 & .79 & .70
acrossICCs: .75 & .70 & .80 & .72 & .67 & .61 & .85 & .33 & .52 & .37 & .94 & .77 & .67
withinICCs: .46 & .43 & .70 & .39 & .57 & .48 & .85 & .23 & .44 & .27 & .93 & .66 & .53
MAEs: .19 & .16 & .38 & .08 & .26 & .19 & .31 & .17 & .22 & .13 & .27 & .27 & .22
MSEs: .19 & .19 & .48 & .04 & .27 & .20 & .26 & .15 & .22 & .13 & .20 & .21 & .21
DISFA_AU_intensity_estimation_threefold    CSN-IR50-stage4


...

,Accuracy,F1 score,Precision,Recall
AU,,,,
1,92.3,50.1,37.0,77.5
2,93.7,44.0,35.6,57.5
4,90.7,73.5,65.0,84.5
6,90.2,54.7,43.1,74.8
9,90.6,41.0,27.8,77.7
12,92.4,75.3,64.8,89.9
25,96.3,93.4,92.8,93.9
26,93.4,67.3,59.6,77.3
average,92.5,62.4,53.2,79.1


accs: 69.1 & 75.3 & 52.4 & 49.0 & 50.3 & 64.7 & 65.9 & 55.5 & 60.3
F1_scores: 21.7 & 18.6 & 38.8 & 23.2 & 14.0 & 42.0 & 61.9 & 27.1 & 30.9
precisions: 12.4 & 10.8 & 24.1 & 13.2 & 7.6 & 26.7 & 44.8 & 15.9 & 19.4
recalls: 86.2 & 65.3 & 99.0 & 97.4 & 96.8 & 99.1 & 99.9 & 94.0 & 92.2
DISFA_AU_detection_threefold    CSN-IR50-fc


,Accuracy,F1 score,Precision,Recall
AU,,,,
1,69.1,21.7,12.4,86.2
2,75.3,18.6,10.8,65.3
4,52.4,38.8,24.1,99.0
6,49.0,23.2,13.2,97.4
9,50.3,14.0,7.6,96.8
12,64.7,42.0,26.7,99.1
25,65.9,61.9,44.8,99.9
26,55.5,27.1,15.9,94.0
average,60.3,30.9,19.4,92.2


accs: 62.6 & 74.4 & 53.3 & 48.4 & 49.0 & 62.8 & 63.9 & 53.6 & 58.5
F1_scores: 18.8 & 17.8 & 39.1 & 23.0 & 13.5 & 40.7 & 60.5 & 25.8 & 29.9
precisions: 10.5 & 10.3 & 24.4 & 13.0 & 7.3 & 25.6 & 43.4 & 15.0 & 18.7
recalls: 86.9 & 64.3 & 98.5 & 97.7 & 95.5 & 99.3 & 99.9 & 91.6 & 91.7
DISFA_AU_detection_threefold    CSN-IR50-output


,Accuracy,F1 score,Precision,Recall
AU,,,,
1,62.6,18.8,10.5,86.9
2,74.4,17.8,10.3,64.3
4,53.3,39.1,24.4,98.5
6,48.4,23.0,13.0,97.7
9,49.0,13.5,7.3,95.5
12,62.8,40.7,25.6,99.3
25,63.9,60.5,43.4,99.9
26,53.6,25.8,15.0,91.6
average,58.5,29.9,18.7,91.7
